# Example Script

In [3]:
import pathlib
import gprofiler
import pandas as pd

def ortho_map(df, name_mapping):

    df.index = df.index.map(name_mapping)

    df.index.name = "symbol"

    df = df.dropna().reset_index().set_index("symbol")
    df = df.reset_index().drop_duplicates(subset=["symbol"]).set_index("symbol")

    return df

for f in pathlib.Path("/home/jhaberbe/Projects/Personal/APR2026/macaque/notebook/deseq2_junior_senior/").glob("*.csv"):
    df = pd.read_csv(f, index_col=0)

gp = gprofiler.GProfiler(return_dataframe=True)

orthologs = gp.orth(
    df.index.tolist(),
    organism="mmulatta",
    target="hsapiens"
)

name_mapping = orthologs.set_index("incoming")["name"].to_dict()

In [4]:
import sys
import pathlib
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("../src/meta-estimators")

In [5]:
from plotting import ForestPlotter
from enrichr import GeneOntologyLibraries
from meta import GeneOntologyDifferentialResults
from ipf import ReweightingPreparer, IterativeProportionalFitting

In [6]:
from tqdm import tqdm

GO_LIBRARIES = {
    "GO_BP": "GO_Biological_Process_2025",
    "GO_MF": "GO_Molecular_Function_2025",
    "GO_CC": "GO_Cellular_Component_2025",
    "Reactome": "Reactome_Pathways_2024",
    "SynGO": "SynGO_2024",
    "WikiPathways": "WikiPathways_2024_Human",
    "LINCS_Chem_Pert": "LINCS_L1000_Chem_Pert_Consensus_Sigs",
    "LINCS_CRISPR": "LINCS_L1000_CRISPR_KO_Consensus_Sigs"
}

GO_TERMS = {
    k: GeneOntologyLibraries().pull_assignment_matrix(k)
    for k in tqdm(GO_LIBRARIES.values())
}

100%|██████████| 8/8 [01:14<00:00,  9.31s/it]


In [ ]:
import decoupler as dc
from statsmodels.stats.multitest import multipletests

output_dir = pathlib.Path("/home/jhaberbe/Projects/Personal/MAY2026/meta-estimators-go-term-enrichment/output")

for f in pathlib.Path("/home/jhaberbe/Projects/Personal/APR2026/macaque/notebook/deseq2_junior_senior/").glob("*.csv"):

    df = pd.read_csv(f, index_col=0)
    df = ortho_map(df, name_mapping)

    (output_dir / f.stem).mkdir(exist_ok=True)

    for gl_folder, gl_name in GO_LIBRARIES.items():
        go_terms = GO_TERMS[gl_name]

        deseq_results, go_terms, row_marginals, column_marginals = ReweightingPreparer \
            .clean(
                go_terms, 
                df
            )

        balanced_matrix = IterativeProportionalFitting() \
            .get_balanced_matrix(
                go_terms, 
                row_marginals, 
                column_marginals,
                tolerance=1e-3
            )

        gene_ontology_results = GeneOntologyDifferentialResults \
            .meta_estimates(
                deseq_results, 
                balanced_matrix
            )

        reject, pvals_corrected, _, _ = multipletests(gene_ontology_results["pval"], method='fdr_bh')

        gene_ontology_results["fdr_bh"] = pvals_corrected

        new_output = (output_dir / f.stem / gl_folder)
        new_output.mkdir(exist_ok=True)

        dc.pl.volcano(
            gene_ontology_results,
            x="metaLFC",
            y="fdr_bh",
            figsize=(15, 10),
            save=str(new_output / "volcano.svg")
        )

        for term in gene_ontology_results.query("fdr_bh < 0.05").sort_values(by="metaLFC").index:
            output = ForestPlotter(deseq_results, balanced_matrix, gene_ontology_results).plot_term(term, cutoff=0.999)
            plt.savefig(new_output / f"{term.replace('/', ' - ')}.svg")
            plt.clf()

        gene_ontology_results.to_csv(new_output / f"{ gl_name }.csv")

/home/jhaberbe/Projects/Personal/MAY2026/meta-estimators-go-term-enrichment/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
IPF Running:   0%|          | 3/1000 [00:00<04:23,  3.79it/s, % Change (Frob Norm)=0.22]